In [1]:
!pip -q install langchain
!pip -q install langchain-community
!pip -q install sentence-transformers
!pip -q install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 63.0 MB/s eta 0:00:00


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


In [7]:
text = """
Machine Learning is a branch of Artificial Intelligence.

Deep Learning is a subset of Machine Learning.

Neural Networks are inspired by the human brain.

Regression predicts continuous values.

Classification predicts labels.

Decision Trees are powerful models.
"""

In [8]:
doc = Document(page_content=text)
print(doc)

page_content='
Machine Learning is a branch of Artificial Intelligence.

Deep Learning is a subset of Machine Learning.

Neural Networks are inspired by the human brain.

Regression predicts continuous values.

Classification predicts labels.

Decision Trees are powerful models.
'


In [9]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=80,
    chunk_overlap=20
)
chunks=splitter.split_documents([doc])

In [10]:
for chunk in chunks:
  print(chunk.page_content)
  print("-"*50)

Machine Learning is a branch of Artificial Intelligence.
--------------------------------------------------
Deep Learning is a subset of Machine Learning.
--------------------------------------------------
Neural Networks are inspired by the human brain.
--------------------------------------------------
Regression predicts continuous values.

Classification predicts labels.
--------------------------------------------------
Decision Trees are powerful models.
--------------------------------------------------


#Embedding Model

In [11]:
embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_6553/1131179023.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model=HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Create VectorDatabase(FAISS)

In [12]:
db=FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [13]:
db

In [14]:
query="What is Deep Learning?"
results=db.similarity_search(
    query,
    k=2
)
#at first query converts into embeddings using embed_model,then calculate cosine similarity between query and chunks ,then returns top k chunks

In [16]:
for doc in results:
  print(doc.page_content)
  print("="*20)

Deep Learning is a subset of Machine Learning.
Machine Learning is a branch of Artificial Intelligence.


In [17]:
query="Which algorithm predicts labels?"
results=db.similarity_search(
    query,
    k=2
)
for doc in results:
  print(doc)
  print("*"*30)

page_content='Regression predicts continuous values.

Classification predicts labels.'
******************************
page_content='Decision Trees are powerful models.'
******************************


In [19]:
query="Artificial Intelligence"
results=db.similarity_search(
    query,
    k=3
)
for i,doc in enumerate(results):

    print("="*60)

    print(f"Result {i+1}")

    print(doc.page_content)

Result 1
Machine Learning is a branch of Artificial Intelligence.
Result 2
Deep Learning is a subset of Machine Learning.
Result 3
Neural Networks are inspired by the human brain.


In [20]:
# pdf-->loader-->chunks-->embedding_model-->364 dim embed-->FAISS (index)-->user query-->embedd-->similarity search-->Top k chunks